# Content-Aware Dynamic Subtitles
### Magic Hour Take-Home

This notebook generates dynamic subtitles that adapt to both the spoken content and the visual composition of a video. It combines local speech transcription with word-level timestamps, semantic caption chunking, deterministic tone-aware styling, expression-aware word emphasis, scene-aware placement, person/object-aware obstruction handling, protected head/face regions, and optional behind-subject compositing.

## How It Works

```text
Video
  |
  +-- Audio
  |     |
  |     v
  |  faster-whisper
  |     |
  |     v
  |  word timestamps
  |     |
  |     v
  |  semantic chunking
  |     |
  |     v
  |  tone + expression analysis
  |
  +-- Frames
        |
        v
     YOLO segmentation
     + clutter/motion analysis
        |
        v
   placement planner
        |
        v
   optional foreground compositing
        |
        v
      renderer
        |
        v
    final video
```

**PLACEMENT FIRST → SUBJECT / HEAD SAFETY → OCCLUSION ONLY AT THE CHOSEN PLACEMENT → RENDER**

The system does not move subtitles toward a person merely to create an occlusion effect. If the already-selected position naturally intersects a safe foreground region, that foreground can be composited back over the text.

In [ ]:
# @title Runtime Check
import platform

try:
    import torch
    CUDA_AVAILABLE = torch.cuda.is_available()
except ImportError:
    CUDA_AVAILABLE = False

SELECTED_DEVICE = "cuda" if CUDA_AVAILABLE else "cpu"
print(f"Python version: {platform.python_version()}")
print(f"CUDA available: {CUDA_AVAILABLE}")
print(f"Selected device: {SELECTED_DEVICE}")

if CUDA_AVAILABLE:
    print("GPU detected: CUDA acceleration available.")
else:
    print("GPU not detected: CPU fallback will be used.")
    print("Processing will be slower but is fully supported.")

## Setup

Clone the project and install its Colab dependencies. This cell is safe to rerun and requires no API key or paid service.

In [ ]:
# @title Install Magic Hour Dynamic Subtitles
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/rahulsiiitm/magic-hour-subtitles.git"
PROJECT_DIR = Path("/content/magic-hour-subtitles")

if PROJECT_DIR.exists():
    print(f"Reusing existing repository: {PROJECT_DIR}")
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements-colab.txt")],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(PROJECT_DIR)],
    check=True,
)
os.chdir(PROJECT_DIR)
print("Setup complete.")

In [ ]:
# @title Verify Installation
import magic_hour_subtitles

print("Package import successful.")
print(f"Loaded from: {magic_hour_subtitles.__file__}")

## Upload Video

Upload one video. If several files are selected, the first supported `.mp4`, `.mov`, `.mkv`, `.webm`, or `.avi` file is used automatically.

In [ ]:
# @title Select Input Video
from google.colab import files
from pathlib import Path
import os

SUPPORTED_VIDEO_EXTENSIONS = {".mp4", ".mov", ".mkv", ".webm", ".avi"}
os.chdir("/content")
uploaded = files.upload()

selected_name = next(
    (name for name in uploaded if Path(name).suffix.lower() in SUPPORTED_VIDEO_EXTENSIONS),
    None,
)
if selected_name is None:
    raise ValueError("No supported video was uploaded. Use MP4, MOV, MKV, WEBM, or AVI.")

INPUT_VIDEO = str(Path("/content") / selected_name)
print(f"Selected input: {selected_name}")

## Configuration

The defaults enable the complete content-aware experience. Diagnostics are off by default; enabling them prints caption tone, selected expressions, placement anchor, person/foreground overlap, head safety, and the behind-subject decision during the same render.

In [ ]:
# @title Demo Options
ENABLE_DYNAMIC_CAPTIONS = True  # @param {type:"boolean"}
ENABLE_SMART_PLACEMENT = True  # @param {type:"boolean"}
ENABLE_BEHIND_SUBJECT = True  # @param {type:"boolean"}
SHOW_DIAGNOSTICS = False  # @param {type:"boolean"}
LANGUAGE = "en"  # @param {type:"string"}
OUTPUT_NAME = "/content/magic_hour_output.mp4"  # @param {type:"string"}

print(f"Language: {LANGUAGE}")
print(f"Output: {OUTPUT_NAME}")

## Generate Subtitles

The existing command-line pipeline performs transcription, analysis, placement, optional foreground compositing, rendering, and final video assembly. The first run may also download the speech and segmentation models.

In [ ]:
# @title Generate Captioned Video
from pathlib import Path
import shlex
import subprocess
import sys

command = [
    sys.executable,
    "-m",
    "magic_hour_subtitles",
    INPUT_VIDEO,
    "-o",
    OUTPUT_NAME,
    "--language",
    LANGUAGE,
]
if ENABLE_DYNAMIC_CAPTIONS:
    command.append("--dynamic-captions")
if ENABLE_SMART_PLACEMENT:
    command.append("--smart-placement")
if ENABLE_BEHIND_SUBJECT:
    command.append("--behind-subject")
if SHOW_DIAGNOSTICS:
    command.append("--caption-diagnostics")

print("Generating subtitles...")
print("This may take a few minutes. GPU acceleration is recommended; CPU fallback is supported.")
print(f"Command: {shlex.join(command)}")
subprocess.run(command, check=True)

output_path = Path(OUTPUT_NAME)
if not output_path.is_file():
    raise RuntimeError(f"Rendering completed without producing the expected output: {OUTPUT_NAME}")

print("\nProcessing complete.")
print(f"Input: {Path(INPUT_VIDEO).name}")
print(f"Output: {output_path.name}")

## Preview Result

In [ ]:
# @title Result
from IPython.display import Video, display

display(Video(OUTPUT_NAME, embed=True))

## Download Result

In [ ]:
# @title Download Captioned Video
from google.colab import files

files.download(OUTPUT_NAME)

## Limitations

- Transcription quality depends on speech clarity.
- Singing and music-heavy audio may produce weaker transcription.
- CPU execution is significantly slower than GPU execution.
- Foreground interactions depend on segmentation quality.
- Tone analysis is deterministic semantic analysis and does not infer a speaker's real emotion.
- Behind-subject compositing is opportunistic and will not activate in every scene.
- Readability and subject safety are prioritized over forcing visual effects.